# 03 Normalize Vital Signs

This notebook transforms raw `observations.csv` into a normalized `vital_signs.csv` dataset. Each measurement becomes one row and is linked to `patient.id` through the raw source patient id.

In [1]:
import pandas as pd
import numpy as np
import uuid
from pathlib import Path

RAW_DATA_DIR = Path('data/raw')
PROCESSED_DATA_DIR = Path('data/processed')

OBSERVATIONS_FILE = RAW_DATA_DIR / 'observations.csv'
PATIENT_FILE = PROCESSED_DATA_DIR / 'patient.csv'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
VITAL_SIGN_CODE_MAP = {
    '8480-6': 'BLOOD_PRESSURE_SYSTOLIC',
    '8462-4': 'BLOOD_PRESSURE_DIASTOLIC',
    '8867-4': 'HEART_RATE',
    '8310-5': 'TEMPERATURE',
    '8302-2': 'HEIGHT',
    '29463-7': 'WEIGHT',
    '39156-5': 'BMI',
    '2339-0': 'GLUCOSE',
    '2093-3': 'CHOLESTEROL',
    '2708-6': 'OXYGEN_SATURATION',
}


def normalize_empty_strings(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace(r'^\s*$', np.nan, regex=True)


def normalize_unit(value: str) -> str:
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    unit_map = {
        'kg': 'kg',
        'g': 'g',
        'cm': 'cm',
        'm': 'm',
        'Cel': 'C',
        'C': 'C',
        'mm[Hg]': 'mmHg',
        '/min': 'beats/min',
        '%': '%',
        'kg/m2': 'kg/m2',
        'mg/dL': 'mg/dL',
        'mmol/L': 'mmol/L',
    }
    return unit_map.get(value, value)

In [3]:
observations_df = pd.read_csv(OBSERVATIONS_FILE)
patient_df = pd.read_csv(PATIENT_FILE)

print('Raw observations shape:', observations_df.shape)
print('Patient table shape:', patient_df.shape)
observations_df.head()

Raw observations shape: (221527, 9)
Patient table shape: (333, 13)


,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
0,2016-08-08T02:57:01Z,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,vital-signs,8302-2,Body Height,119.3,cm,numeric
1,2019-09-10T08:06:56Z,2d68ad16-268a-478c-1f84-d0f1976e1a46,2d68ad16-268a-478c-73b1-09735410738b,vital-signs,8302-2,Body Height,50.7,cm,numeric
2,2019-09-10T08:06:56Z,2d68ad16-268a-478c-1f84-d0f1976e1a46,2d68ad16-268a-478c-73b1-09735410738b,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,0.0,{score},numeric
3,2016-08-08T02:57:01Z,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,1.0,{score},numeric
4,2016-08-08T02:57:01Z,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,vital-signs,29463-7,Body Weight,21.3,kg,numeric


In [4]:
rename_map = {
    'DATE': 'measured_at',
    'PATIENT': 'source_patient_id',
    'ENCOUNTER': 'source_encounter_id',
    'CODE': 'observation_code',
    'DESCRIPTION': 'description',
    'VALUE': 'value',
    'UNITS': 'unit',
    'TYPE': 'source_type',
}

vitals_df = observations_df.rename(columns=rename_map).copy()
vitals_df = normalize_empty_strings(vitals_df)

required_columns = [
    'measured_at',
    'source_patient_id',
    'source_encounter_id',
    'observation_code',
    'description',
    'value',
    'unit',
    'source_type',
]

existing_columns = [col for col in required_columns if col in vitals_df.columns]
vitals_df = vitals_df[existing_columns].copy()
vitals_df.head()

,measured_at,source_patient_id,source_encounter_id,observation_code,description,value,unit,source_type
0,2016-08-08T02:57:01Z,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,8302-2,Body Height,119.3,cm,numeric
1,2019-09-10T08:06:56Z,2d68ad16-268a-478c-1f84-d0f1976e1a46,2d68ad16-268a-478c-73b1-09735410738b,8302-2,Body Height,50.7,cm,numeric
2,2019-09-10T08:06:56Z,2d68ad16-268a-478c-1f84-d0f1976e1a46,2d68ad16-268a-478c-73b1-09735410738b,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,0.0,{score},numeric
3,2016-08-08T02:57:01Z,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,1.0,{score},numeric
4,2016-08-08T02:57:01Z,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,29463-7,Body Weight,21.3,kg,numeric


In [5]:
vitals_df = vitals_df[vitals_df['observation_code'].astype(str).isin(VITAL_SIGN_CODE_MAP.keys())].copy()

vitals_df['vital_type'] = vitals_df['observation_code'].astype(str).map(VITAL_SIGN_CODE_MAP)
vitals_df['measured_at'] = pd.to_datetime(vitals_df['measured_at'], errors='coerce')
vitals_df['value'] = pd.to_numeric(vitals_df['value'], errors='coerce')
vitals_df['unit'] = vitals_df['unit'].apply(normalize_unit)

vitals_df = vitals_df.dropna(subset=['source_patient_id', 'vital_type', 'measured_at', 'value']).copy()

print('Filtered vital signs shape:', vitals_df.shape)
vitals_df.head()

Filtered vital signs shape: (30390, 9)


,measured_at,source_patient_id,source_encounter_id,observation_code,description,value,unit,source_type,vital_type
0,2016-08-08 02:57:01+00:00,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,8302-2,Body Height,119.3,cm,numeric,HEIGHT
1,2019-09-10 08:06:56+00:00,2d68ad16-268a-478c-1f84-d0f1976e1a46,2d68ad16-268a-478c-73b1-09735410738b,8302-2,Body Height,50.7,cm,numeric,HEIGHT
4,2016-08-08 02:57:01+00:00,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,29463-7,Body Weight,21.3,kg,numeric,WEIGHT
5,2016-08-08 02:57:01+00:00,5e688e99-61b3-5c88-3f60-21df8aaced27,5e688e99-61b3-5c88-b836-699c374fecc4,39156-5,Body mass index (BMI) [Ratio],15.0,kg/m2,numeric,BMI
6,2019-09-10 08:06:56+00:00,2d68ad16-268a-478c-1f84-d0f1976e1a46,2d68ad16-268a-478c-73b1-09735410738b,29463-7,Body Weight,3.2,kg,numeric,WEIGHT


In [6]:
vital_signs_df = vitals_df.merge(
    patient_df[['id', 'source_patient_id']],
    on='source_patient_id',
    how='inner'
)

vital_signs_df = vital_signs_df.rename(columns={'id': 'patient_id'})
vital_signs_df.insert(0, 'id', [str(uuid.uuid4()) for _ in range(len(vital_signs_df))])

vital_signs_df = vital_signs_df[[
    'id',
    'patient_id',
    'vital_type',
    'value',
    'unit',
    'measured_at',
    'source_patient_id',
    'source_encounter_id',
    'observation_code',
    'description',
]].copy()

vital_signs_df = vital_signs_df.sort_values(by=['patient_id', 'measured_at', 'vital_type']).reset_index(drop=True)
vital_signs_df.head()

,id,patient_id,vital_type,value,unit,measured_at,source_patient_id,source_encounter_id,observation_code,description
0,c013dc22-9454-4549-9736-a776cd74d8ed,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,BLOOD_PRESSURE_DIASTOLIC,91.0,mmHg,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,8462-4,Diastolic Blood Pressure
1,14b02d20-c081-4518-a221-a5fe9b30f033,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,BLOOD_PRESSURE_SYSTOLIC,148.0,mmHg,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,8480-6,Systolic Blood Pressure
2,e7a5d07b-aab6-4094-aadb-40c6ccd569a8,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,HEART_RATE,81.0,beats/min,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,8867-4,Heart rate
3,f16173a8-2796-48b5-af3a-a6f8f0f6d774,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,HEIGHT,52.7,cm,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,8302-2,Body Height
4,59a79678-d7bf-42ee-ac89-7fcfbce40327,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,WEIGHT,3.7,kg,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,29463-7,Body Weight


In [7]:
print('Vital signs null counts:')
print(vital_signs_df.isna().sum())

print('\nVital sign types:')
print(vital_signs_df['vital_type'].value_counts())

print('\nRows without patient_id:', vital_signs_df['patient_id'].isna().sum())
print('Duplicate rows:', vital_signs_df.duplicated(subset=['patient_id', 'vital_type', 'value', 'measured_at']).sum())

Vital signs null counts:
id                     0
patient_id             0
vital_type             0
value                  0
unit                   0
measured_at            0
source_patient_id      0
source_encounter_id    0
observation_code       0
description            0
dtype: int64

Vital sign types:
vital_type
BLOOD_PRESSURE_DIASTOLIC    4415
BLOOD_PRESSURE_SYSTOLIC     4415
WEIGHT                      4379
HEART_RATE                  4356
HEIGHT                      4235
BMI                         3910
GLUCOSE                     2517
CHOLESTEROL                 1506
TEMPERATURE                  397
OXYGEN_SATURATION            260
Name: count, dtype: int64

Rows without patient_id: 0
Duplicate rows: 9


In [8]:
vital_signs_output_file = PROCESSED_DATA_DIR / 'vital_signs.csv'
vital_signs_df.to_csv(vital_signs_output_file, index=False)

print('Exported:', vital_signs_output_file)

Exported: data\processed\vital_signs.csv


In [9]:
display(vital_signs_df.head(20))

,id,patient_id,vital_type,value,unit,measured_at,source_patient_id,source_encounter_id,observation_code,description
0,c013dc22-9454-4549-9736-a776cd74d8ed,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,BLOOD_PRESSURE_DIASTOLIC,91.0,mmHg,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,8462-4,Diastolic Blood Pressure
1,14b02d20-c081-4518-a221-a5fe9b30f033,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,BLOOD_PRESSURE_SYSTOLIC,148.0,mmHg,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,8480-6,Systolic Blood Pressure
2,e7a5d07b-aab6-4094-aadb-40c6ccd569a8,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,HEART_RATE,81.0,beats/min,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,8867-4,Heart rate
3,f16173a8-2796-48b5-af3a-a6f8f0f6d774,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,HEIGHT,52.7,cm,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,8302-2,Body Height
4,59a79678-d7bf-42ee-ac89-7fcfbce40327,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,WEIGHT,3.7,kg,2024-07-04 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-cbfa-81e28ba3da9b,29463-7,Body Weight
5,a12a0d03-8c1d-41a5-a8db-20ef7fe5add4,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,BLOOD_PRESSURE_DIASTOLIC,86.0,mmHg,2024-08-08 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-78da-33e37861355f,8462-4,Diastolic Blood Pressure
6,f72d3f52-56c1-43f5-a8a0-4ababe7f0c49,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,BLOOD_PRESSURE_SYSTOLIC,140.0,mmHg,2024-08-08 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-78da-33e37861355f,8480-6,Systolic Blood Pressure
7,266960aa-3ba3-45a0-841a-2f572b5a015b,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,HEART_RATE,88.0,beats/min,2024-08-08 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-78da-33e37861355f,8867-4,Heart rate
8,f08a8ef5-cd74-410b-b840-481e6e3bebc3,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,HEIGHT,56.3,cm,2024-08-08 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-78da-33e37861355f,8302-2,Body Height
9,00aaa53b-a990-4093-8131-f4ba9ca03eac,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,WEIGHT,4.4,kg,2024-08-08 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-78da-33e37861355f,29463-7,Body Weight
